# Acquirium client reference

This is the reference guide for the acquirium client: the query interface feature by feature, with the internals (`to_sparql`, text resolution) shown along the way. If you are new, start with `quickstart.ipynb`.

To get support reach out to [Mete](mailto:saka@mines.edu)!

## Setup
0. Follow the steps in [deployments/WATERTAP/README.md](../../deployments/WATERTAP/README.md) to install requirements
1. Start the server with a config: `acquirium server --config deployments/WATERTAP/scripts/acquirium.toml`
2. Connect:

In [1]:
from datetime import datetime, timedelta, timezone
from acquirium import Acquirium

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

## Find entities by class
`entity()` accepts a URI or a natural-language string. The alias defaults to what you typed; `alias=` or `.alias()` renames it.

In [3]:
q = acq.query().entity("Pump")
q.metadata()

Pump
str
"""wbs:intake"""
"""wbs:P1"""
"""wbs:P2"""


### How strings become URIs
Every string is resolved server-side by an embedding matcher. `resolve()` shows what a string resolves to — check it when a query returns something unexpected, and pass exact URIs when correctness matters (the top match is not always the intended one):

In [4]:
acq.client.resolve("salt", kind="class", top_k=3)

[{'uri': 'urn:nawi-water-ontology#Salt-NaCl',
  'kind': 'class',
  'label': 'Salt-NaCl',
  'score': 0.8903464674949646,
  'matched_surface': 'salt na cl',
  'match_stage': 'semantic',
  'related': []},
 {'uri': 'urn:nawi-water-ontology#Constituent-Salt',
  'kind': 'class',
  'label': 'Constituent-Salt',
  'score': 0.7883857488632202,
  'matched_surface': 'constituent salt',
  'match_stage': 'semantic',
  'related': []},
 {'uri': 'urn:nawi-water-ontology#pHAdjuster-Lime',
  'kind': 'class',
  'label': 'Lime',
  'score': 0.757838785648346,
  'matched_surface': 'lime',
  'match_stage': 'semantic',
  'related': []}]

## Follow relationships
`related()` adds a neighbour. By default it walks any non-hidden predicate up to 3 hops and keeps the nearest match. `via=` restricts traversal to one predicate or a list of them, `max_depth=` sets the reach, and `direction="upstream"/"downstream"` walks the S223 piping topology instead. Strings and URIs are both accepted.

In [6]:
q = (
    acq.query().entity("Pump")
    .related("tank")
)
q.metadata()


Pump,tank
str,str
"""wbs:intake""","""wbs:ferric-chloride-addition"""
"""wbs:P2""","""wbs:storage-tank-2"""
"""wbs:P1""","""wbs:storage-tank-2"""
"""wbs:P1""","""wbs:anti-scalant-addition"""


## Attach data nodes
`measurement()` adds the data-bearing points of the current node: its own properties plus the ones hanging off its connection points. On an empty query it matches every registered stream in the plant.

In [8]:
q = acq.query().entity("Pump").measurement()
q.metadata()

Pump,Pump_data
str,str
"""wbs:P1""","""wbs:P1-out-pressure"""
"""wbs:P1""","""wbs:P1-mechanical-power"""
"""wbs:P2""","""wbs:P2-mechanical-power"""
"""wbs:intake""","""wbs:intake-in-tss-concentratio…"
"""wbs:intake""","""wbs:intake-in-tds-concentratio…"
"""wbs:intake""","""wbs:intake-in-flow-rate"""
"""wbs:P1""","""wbs:P1-efficiency"""
"""wbs:P2""","""wbs:P2-efficiency"""
"""wbs:intake""","""wbs:intake-in-toc-concentratio…"


In [10]:
acq.query().measurement().metadata()

data
str
"""wbs:RO-membrane-area"""
"""wbs:RO-out-flow-mass-water"""
"""ns3:point_6"""
"""wbs:P2-efficiency"""
"""ns3:point_9"""
…
"""wbs:conn-cartridge-filtration-…"
"""ns3:point_4"""
"""wbs:storage-tank-3-out-flow-ra…"


## Filter data nodes
`where()` filters the node the pointer is on. Strings are resolved via the text matcher. The same keywords work inline on `entity()`, `related()` and `measurement()`.

In [12]:
q = acq.query().measurement().where(quantity_kind="Pressure")
q.metadata()

data
str
"""wbs:conn-cartridge-filtration-…"
"""wbs:RO-out-retentate-pressure"""
"""wbs:RO-in-pressure"""
"""wbs:P1-out-pressure"""
"""wbs:RO-out-pressure"""
"""wbs:PXR-brine-out-pressure"""


In [14]:
q = acq.query().measurement().where(unit="kg/s")
q.metadata()

data
str
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:RO-in-flow-mass-water"""
"""wbs:RO-out-flow-mass-water"""
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:PXR-brine-out-flow-mass-td…"
"""wbs:RO-out-flow-mass-tds"""
"""wbs:conn-cartridge-filtration-…"
"""wbs:conn-cartridge-filtration-…"
"""wbs:PXR-brine-out-flow-mass-wa…"


In [16]:
q = acq.query().measurement().where(unit="kg/s",substance="constituent salt")
q.metadata()

data
str
"""wbs:RO-in-flow-mass-tds"""
"""wbs:RO-out-flow-mass-tds"""
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:conn-cartridge-filtration-…"
"""wbs:PXR-brine-out-flow-mass-td…"


`unit`, `substance`, `quantity_kind` and `medium` are attributes, not methods: one vocabulary shared by `where()`, `include()`, `options()` and the inline keywords. `medium` covers both predicates that carry it — `s223:ofMedium` on a property and `s223:hasMedium` on a connection point — so one filter finds the brine points wherever the medium is declared. Wrap a value in `Not()` to exclude it instead.

In [ ]:
q_brine = acq.query().measurement().where(medium="brine")
q_brine.metadata()

## Inspect the query
Every query compiles to SPARQL against the server's graph — nothing is hidden. `to_sparql()` returns the compiled query without running it (check it when a result surprises you), and `metadata()` returns the full result as a polars DataFrame (`include_internals=True` keeps the ref and unit columns).

In [ ]:
q = acq.query().measurement().where(unit="kg/s", substance="constituent salt")

In [ ]:
print(q.to_sparql())

In [ ]:
df_meta = q.metadata()
df_meta

## Pull timeseries
`dataframe()` returns a polars frame. `shape="wide"` puts each data node in its own column, `"narrow"` is long-form. The two entry points default differently: `Query.dataframe()` is narrow with `cast_value="str"`, `DataObject.dataframe()` is wide. Pass `shape=` explicitly and the question does not arise.

In [21]:
q = acq.query().measurement().where(unit="kg/s",substance="constituent salt")


end = datetime.now(tz=timezone.utc)
start = end - timedelta(hours=10)

df = q.dataframe(start=start, end=end, shape="wide", cast_value="float")
df.head()

time,data__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,data__wbs:PXR-brine-out-flow-mass-tds,data__wbs:RO-in-flow-mass-tds,data__wbs:RO-out-flow-mass-tds,data__wbs:RO-out-retentate-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-04 20:53:02.260671 UTC,11.636159,11.606711,11.636159,0.029447,11.606711
2026-08-04 20:53:20.200666 UTC,11.128819,11.099154,11.128819,0.029665,11.099154
2026-08-04 20:53:39.151741 UTC,11.319533,11.289939,11.319533,0.029594,11.289939
2026-08-04 20:54:03.991986 UTC,11.335039,11.30535,11.335039,0.029689,11.30535
2026-08-04 20:54:19.432224 UTC,11.687578,11.658163,11.687578,0.029415,11.658163


In [22]:
q.dataframe(limit=1, order='desc', shape="wide")

time,data__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,data__wbs:PXR-brine-out-flow-mass-tds,data__wbs:RO-in-flow-mass-tds,data__wbs:RO-out-flow-mass-tds,data__wbs:RO-out-retentate-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-05 00:04:20.086642 UTC,10.675185,10.645048,10.675185,0.030136,10.645048


## Structured access via DataObject
`data()` returns an object keyed by alias for quick lookups.

In [23]:
import polars as pl


data = q.data(start=start, end=end, cast_value="float")
data.dataframe()

time,data__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,data__wbs:PXR-brine-out-flow-mass-tds,data__wbs:RO-in-flow-mass-tds,data__wbs:RO-out-flow-mass-tds,data__wbs:RO-out-retentate-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-04 20:53:02.260671 UTC,11.636159,11.606711,11.636159,0.029447,11.606711
2026-08-04 20:53:20.200666 UTC,11.128819,11.099154,11.128819,0.029665,11.099154
2026-08-04 20:53:39.151741 UTC,11.319533,11.289939,11.319533,0.029594,11.289939
2026-08-04 20:54:03.991986 UTC,11.335039,11.30535,11.335039,0.029689,11.30535
2026-08-04 20:54:19.432224 UTC,11.687578,11.658163,11.687578,0.029415,11.658163
…,…,…,…,…,…
2026-08-05 00:03:20.418760 UTC,10.80747,10.777669,10.80747,0.029801,10.777669
2026-08-05 00:03:35.501514 UTC,10.571054,10.540776,10.571054,0.030278,10.540776
2026-08-05 00:03:49.888609 UTC,10.815503,10.785598,10.815503,0.029905,10.785598


## Inspect units
`units()` returns the effective QUDT unit URI per data alias.

In [24]:
data.units()

{'data': 'http://qudt.org/vocab/unit/KiloGM-PER-SEC'}

## Convert units
`convert_to(target)` accepts any QUDT-recognized identifier (URI, label, symbol, UCUM code). The returned DataObject has values converted and `units()` updated.

In [25]:
data = data.convert_to("kg/min")
data.dataframe().head()

time,data__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,data__wbs:PXR-brine-out-flow-mass-tds,data__wbs:RO-in-flow-mass-tds,data__wbs:RO-out-flow-mass-tds,data__wbs:RO-out-retentate-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-04 20:53:02.260671 UTC,698.16953,696.402685,698.16953,1.766845,696.402685
2026-08-04 20:53:20.200666 UTC,667.72916,665.949253,667.72916,1.779907,665.949253
2026-08-04 20:53:39.151741 UTC,679.171988,677.396365,679.171988,1.775623,677.396365
2026-08-04 20:54:03.991986 UTC,680.102332,678.321004,680.102332,1.781327,678.321004
2026-08-04 20:54:19.432224 UTC,701.254703,699.48978,701.254703,1.764924,699.48978


### Systems

Systems are logical groupings of equipment and junctions (and other systems) in S223 ontology (parent ontology of WaTr)

The systems in the model are:

In [27]:
q = acq.query().entity(cls="system")
q.metadata()

system
str
"""wbs:posttreatment-system"""
"""wbs:desalination-system"""
"""wbs:pretreatment-system"""
"""wbs:seawater-ro-plant"""


The systems are hierarchically organized as:

In [29]:
q = acq.query().entity(cls="system").related("system").alias("subsystem")
q.metadata()

system,subsystem
str,str
"""wbs:seawater-ro-plant""","""wbs:desalination-system"""
"""wbs:seawater-ro-plant""","""wbs:posttreatment-system"""
"""wbs:seawater-ro-plant""","""wbs:pretreatment-system"""


We can see how many equipment we have in each system:

In [31]:
q = acq.query().entity(cls="System").related("Equipment")
q.metadata()

System,Equipment
str,str
"""wbs:desalination-system""","""wbs:RO"""
"""wbs:pretreatment-system""","""wbs:ferric-chloride-addition"""
"""wbs:seawater-ro-plant""","""wbs:P1"""
"""wbs:pretreatment-system""","""wbs:static-mixer"""
"""wbs:seawater-ro-plant""","""wbs:backwash-handling"""
…,…
"""wbs:seawater-ro-plant""","""wbs:media-filtration"""
"""wbs:seawater-ro-plant""","""wbs:uv-aop"""
"""wbs:posttreatment-system""","""wbs:co2-addition"""


In [32]:
q.metadata().group_by("System").agg(pl.col("Equipment").count().alias("equipment_count")).sort("equipment_count",descending=True)

System,equipment_count
str,u32
"""wbs:seawater-ro-plant""",18
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


These are the equipment directly a member of each system.

`via="has member"` with `nearest=False` follows membership all the way down, so equipment nested in a subsystem counts too:

In [34]:
q = acq.query().entity(cls="System").related("Equipment",via="has member",nearest=False)
q.metadata().group_by("System").agg(pl.col("Equipment").count().alias("equipment_count")).sort("equipment_count",descending=True)

System,equipment_count
str,u32
"""wbs:seawater-ro-plant""",18
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


Let's find the pumps in a specific system:


In [36]:
q= acq.query().entity(uri='wbs:pretreatment-system').alias("system").related("pump")
q.metadata()

system,pump
str,str
"""wbs:pretreatment-system""","""wbs:intake"""


Let's find all the pumps and their data

In [38]:
q = acq.query().entity("pump").measurement()
q.metadata().head()

pump,pump_data
str,str
"""wbs:intake""","""wbs:intake-in-tds-concentratio…"
"""wbs:P1""","""wbs:P1-efficiency"""
"""wbs:P1""","""wbs:P1-mechanical-power"""
"""wbs:P2""","""wbs:P2-efficiency"""
"""wbs:intake""","""wbs:intake-in-tss-concentratio…"


Let's find all the data generating entites within a system:

In [40]:
q = (acq.query().entity(uri = 'wbs:pretreatment-system').drop()
     .related('equipment').measurement(alias="sensor").include("unit"))
q.metadata()

equipment,sensor,sensor.unit
str,str,str
"""wbs:intake""","""wbs:intake-in-flow-rate""","""None"""
"""wbs:intake""","""wbs:intake-in-tss-concentratio…","""unit:MilliGM-PER-L"""
"""wbs:cartridge-filtration""","""wbs:cartridge-filtration-out-t…","""unit:KiloGM-PER-M3"""
"""wbs:intake""","""wbs:intake-in-toc-concentratio…","""unit:KiloGM-PER-M3"""
"""wbs:intake""","""wbs:intake-in-tds-concentratio…","""unit:KiloGM-PER-M3"""
